# EcoSort Waste Management Assistant
# Module 8 Summative Lab

## Overview

You are a data scientist at "EcoSort," a technology company that specializes in developing AI solutions for waste management. EcoSort has partnered with Metro City's waste management department to develop an intelligent waste management assistant that can help residents properly dispose of waste items so less time is spent sorting material at facilities.

This assistant needs to:

1. Identify waste materials from images uploaded by residents (CNN)
2. Classify waste items based on text descriptions provided by residents (RNN/Transformer)
3. Generate specific recycling instructions based on identified waste type and city policies (Generative Transformer with RAG)

Your task is to build this integrated system using the RealWaste dataset along with generated text data that simulates real-world waste management operations.

## Part 1: Dataset Exploration and Preparation

In this section, you will explore and prepare the datasets for your models.

### 1.1 Load and Explore the RealWaste Dataset

**Setup:** run this install cell once. It adds the libraries for sentence embeddings and the text generator. Skip any you already have.

In [ ]:
# '!' runs a terminal command from inside a notebook cell
!pip install -q sentence-transformers transformers seaborn scikit-learn

In [ ]:
# Import necessary libraries
import os, re, json, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

These are the toolboxes the notebook uses: pandas for tables, matplotlib/seaborn for charts, TensorFlow for the image model and scikit-learn for the text model and evaluation. The seeds make results repeatable.

In [ ]:
import pathlib

# Location of the RealWaste folder (the r before the quotes = "raw string" so Windows backslashes work).
# On Google Colab, change this to the path of the folder in your Drive instead.
data_dir = pathlib.Path(r"realwaste\realwaste-main\RealWaste")

# Check the path is correct (should print True and list 9 folders)
print("Folder exists:", data_dir.exists())

# 1) Dataset structure: count images in each category folder
counts = {}
for folder in sorted(data_dir.glob('*')):
    if folder.is_dir():
        counts[folder.name] = len(list(folder.glob('*.jpg'))) + len(list(folder.glob('*.png')))

counts_df = pd.DataFrame(list(counts.items()), columns=['category', 'num_images'])
print(counts_df)
print("Total images:", counts_df['num_images'].sum())

# 2) Distribution of waste categories (bar chart)
plt.figure(figsize=(9, 4))
sns.barplot(data=counts_df, x='num_images', y='category')
plt.title('Images per waste category')
plt.show()

# 3) Image characteristics: show one random image per category
plt.figure(figsize=(12, 10))
for i, folder in enumerate(sorted([f for f in data_dir.glob('*') if f.is_dir()])):
    img_path = random.choice(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
    img = Image.open(img_path)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(f"{folder.name}\n{img.size}")   # img.size = (width, height)
    plt.axis('off')
plt.tight_layout()
plt.show()

# Check resolution / colour mode on a random sample of 200 images
all_imgs = list(data_dir.glob('*/*.jpg')) + list(data_dir.glob('*/*.png'))
sample = random.sample(all_imgs, min(200, len(all_imgs)))
sizes = Counter(Image.open(p).size for p in sample)
modes = Counter(Image.open(p).mode for p in sample)
print("Image sizes found in sample:", sizes.most_common(5))
print("Colour modes found in sample:", modes)

**What this does:** counts images per class, plots the class balance, shows one example per class, and checks image sizes and colour modes (e.g. `RGB`) to see whether resizing is needed.

**Observations (write your own after running):** Are the classes balanced? Plastic and Metal have far more images than Textile Trash. Are the backgrounds consistent? Are there odd image sizes?

### 1.2 Explore Text Datasets

In [ ]:
df = pd.read_csv('waste_descriptions.csv')

print("Shape (rows, columns):", df.shape)
display(df.head())

print("\nMissing values per column:")
print(df.isna().sum())   # common_confusion is empty for about half the rows, and that's fine

print("\nDuplicate descriptions:", df['description'].duplicated().sum())

# Category distribution
print("\nCategory counts:")
print(df['category'].value_counts())
plt.figure(figsize=(9, 4))
sns.countplot(data=df, y='category', order=df['category'].value_counts().index)
plt.title('Descriptions per category')
plt.show()

# Vocabulary and structure
df['word_count'] = df['description'].str.split().str.len()
print("\nWords per description:")
print(df['word_count'].describe())

all_words = ' '.join(df['description'].str.lower()).split()
print("\nUnique words (vocabulary size):", len(set(all_words)))
print("20 most common words:", Counter(all_words).most_common(20))

**What this does:** loads the CSV and checks its size, missing values and duplicates. It also checks the number of examples per category, description length and vocabulary, which guide the text-processing choices later.

In [ ]:
# The policy file is JSON (not CSV), so we use json.load and turn it into a table
with open('waste_policy_documents.json', 'r') as f:
    policy_docs = json.load(f)

policy_df = pd.DataFrame(policy_docs)
print("Number of policy documents:", len(policy_df))
display(policy_df[['policy_id', 'policy_type', 'categories_covered', 'jurisdiction']])

# How long is each document, and how many categories does it cover?
policy_df['num_words'] = policy_df['document_text'].str.split().str.len()
policy_df['num_categories'] = policy_df['categories_covered'].apply(len)
print(policy_df[['policy_type', 'num_words', 'num_categories']])

# Read one single-category document and one multi-category document
print("\n--- Example: single-category document ---")
print(policy_df.loc[0, 'document_text'])
print("\n--- Example: multi-category document ---")
print(policy_df.loc[10, 'document_text'])

**What this does:** shows how the policies are organised. Documents 1-9 each cover one category with sections such as "Acceptable Items" and "Non-Acceptable Items". Documents 10-14 bundle several categories into one file, which affects how we split them for retrieval.

### 1.3 Create Data Pipelines

In [ ]:
# Run this code to setup the images properly into train, validation, and test sets
# Set your data directory path - update this with your actual path
import pathlib
# data_dir was already set in Section 1.1 above (update the path there if needed)

# Parameters
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

# Calculate the total number of classes automatically from the directory structure
num_classes = len([item for item in data_dir.glob('*') if item.is_dir()])
print(f"Number of classes: {num_classes}")

# List all class folders
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"Class names: {class_names}")

# Count all images
image_count = len(list(data_dir.glob('*/*.jpg'))) + len(list(data_dir.glob('*/*.png')))
print(f"Total images found: {image_count}")

# Create a dataset using tf.keras.utils.image_dataset_from_directory
# This will automatically split the data into training and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=True
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=True
)

# Create a separate test dataset by taking part of the validation set
# First, let's get the number of batches in the validation set
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print(f"Number of training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Number of validation batches: {tf.data.experimental.cardinality(validation_ds)}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset)}")

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# --- Step 1: Text cleaning ---
def clean_text(text):
    """Lowercase, remove punctuation/numbers, and tidy up spaces."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)    # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip() # collapse repeated spaces
    return text

text_df = pd.read_csv('waste_descriptions.csv')
text_df = text_df.drop_duplicates(subset='description').reset_index(drop=True)  # remove repeats
text_df['clean_description'] = text_df['description'].apply(clean_text)

# --- Step 2: Turn category names into numbers ---
label_encoder = LabelEncoder()
text_df['label'] = label_encoder.fit_transform(text_df['category'])
print("Label mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

# --- Step 3: Split into train and test (80/20) ---
# stratify keeps the same category proportions in both sets
train_df, test_df = train_test_split(
    text_df, test_size=0.2, random_state=42, stratify=text_df['label']
)
print("Train size:", len(train_df), "| Test size:", len(test_df))

# --- Step 4: Tokenization + features (TF-IDF) ---
# TF-IDF gives each word a score for how informative it is.
# ngram_range=(1, 2) also looks at word pairs such as "plastic bottle".
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train = tfidf.fit_transform(train_df['clean_description'])  # learn vocabulary on TRAIN only
X_test = tfidf.transform(test_df['clean_description'])        # reuse it on test (avoids leakage)
y_train = train_df['label']
y_test = test_df['label']
print("Feature matrix shape:", X_train.shape)

**What this does:** cleans the text, converts categories to numbers, splits 80/20 and turns each description into a TF-IDF vector. The vectorizer is fitted on the training data only so no test information leaks into training.

In [ ]:
from sentence_transformers import SentenceTransformer

# --- Step 1: Split each policy into small "chunks" ---
# Small chunks retrieve better than whole documents. We split on blank lines
# and tag each chunk with the waste category it talks about.
all_categories = sorted(text_df['category'].unique())

def find_chunk_category(block, policy_categories):
    first_line = block.strip().split('\n')[0].upper()
    for cat in all_categories:
        if cat.upper() in first_line:          # e.g. "PLASTIC GUIDELINES:"
            return cat
    if len(policy_categories) == 1:            # single-category policy -> that category
        return policy_categories[0]
    return 'General'                           # general rules (e.g. "GENERAL REQUIREMENTS")

chunks = []
for doc in policy_docs:
    for block in doc['document_text'].split('\n\n'):
        if len(block.strip()) < 20:
            continue
        chunks.append({
            'text': f"{doc['policy_type']}: {block.strip()}",
            'category': find_chunk_category(block, doc['categories_covered']),
            'policy_type': doc['policy_type'],
            'source': 'policy'
        })

# --- Step 2: Add the disposal instructions from the CSV as extra knowledge ---
for _, row in df[['category', 'disposal_instruction']].drop_duplicates().iterrows():
    chunks.append({
        'text': f"Disposal tip for {row['category']}: {row['disposal_instruction']}",
        'category': row['category'],
        'policy_type': 'Disposal Instructions (CSV)',
        'source': 'csv'
    })

chunk_df = pd.DataFrame(chunks)
print("Total chunks:", len(chunk_df))
print(chunk_df['category'].value_counts())

# --- Step 3: Create embeddings (numbers that capture meaning) ---
embedder = SentenceTransformer('all-MiniLM-L6-v2')   # small, fast model
chunk_embeddings = embedder.encode(chunk_df['text'].tolist(), normalize_embeddings=True)
print("Embedding matrix shape:", chunk_embeddings.shape)

**What this does:** breaks the policies into small tagged chunks, adds the CSV disposal tips, and uses a sentence-embedding model to turn each chunk into a vector so we can later find the chunks most similar to a question.

## Part 2: Waste Material Classification with CNN

In this section, you will build a CNN model to classify waste materials from images.

### 2.1 Preprocess Images

In [ ]:
# Data augmentation: random changes that make the model more robust.
# These layers only act during training, not when predicting.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

# MobileNetV2 expects pixel values between -1 and 1 (images start at 0-255)
preprocess = layers.Rescaling(1./127.5, offset=-1, name='rescale')

# Quick visual check of the augmentation on one training image
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 4))
    for i in range(4):
        aug = data_augmentation(tf.expand_dims(images[0], 0), training=True)
        plt.subplot(1, 4, i + 1)
        plt.imshow(aug[0].numpy().astype('uint8'))
        plt.axis('off')
    plt.suptitle(f"Augmented versions of one '{class_names[np.argmax(labels[0])]}' image")
    plt.show()

**What this does:** the pipeline above already resizes images to 224x224 and batches them. Here we add augmentation and the pixel rescaling MobileNet needs. Both are built into the model, so they also apply automatically when predicting.

### 2.2 Implement CNN Model with Transfer Learning

In [ ]:
# Transfer learning: reuse MobileNetV2, which was already trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,        # remove its original 1000-class output layer
    weights='imagenet'
)
base_model.trainable = False  # freeze it so only our new layers learn at first

cnn_model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    data_augmentation,
    preprocess,
    base_model,
    layers.GlobalAveragePooling2D(),            # shrink feature maps to one vector
    layers.Dropout(0.3),                        # regularization: randomly switch off 30% of units
    layers.Dense(num_classes, activation='softmax')  # one probability per category
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',   # matches label_mode='categorical'
    metrics=['accuracy']
)
cnn_model.summary()

**What this does:** uses MobileNetV2 as a ready-made image feature extractor and adds a small classification head for the 9 categories. The loss is `categorical_crossentropy` because the labels are one-hot encoded. MobileNetV2 is chosen because it is light, fast and accurate enough for this dataset.

### 2.3 Train and Evaluate the Model

In [ ]:
# Callbacks: stop early if validation stops improving, and lower the learning rate on plateaus
cnn_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

history = cnn_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=cnn_callbacks
)

def plot_history(hist, title=''):
    """Plot accuracy and loss for training vs validation."""
    plt.figure(figsize=(11, 4))
    plt.subplot(1, 2, 1)
    plt.plot(hist.history['accuracy'], label='train')
    plt.plot(hist.history['val_accuracy'], label='validation')
    plt.title(f'{title} Accuracy'); plt.xlabel('Epoch'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(hist.history['loss'], label='train')
    plt.plot(hist.history['val_loss'], label='validation')
    plt.title(f'{title} Loss'); plt.xlabel('Epoch'); plt.legend()
    plt.show()

plot_history(history, 'Stage 1:')

**What this does:** trains for up to 10 epochs and stops early if validation loss stops improving. The curves show overfitting: if training accuracy keeps rising while validation accuracy stalls, the model is memorising.

In [ ]:
def evaluate_cnn(model, dataset, title=''):
    """Predict on a dataset, print metrics, and draw a confusion matrix."""
    y_true, y_pred = [], []
    # We collect labels and predictions in the SAME loop so they always line up
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(probs, axis=1))
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    print(f"{title} Test accuracy: {acc:.3f}\n")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{title} Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.show()

    # Error analysis: which true->predicted pairs are confused most often?
    errors = Counter((class_names[t], class_names[p])
                     for t, p in zip(y_true, y_pred) if t != p)
    print("Most common mistakes (true -> predicted):")
    for (t, p), n in errors.most_common(5):
        print(f"  {t} -> {p}: {n}")
    return acc

stage1_acc = evaluate_cnn(cnn_model, test_dataset, 'Stage 1:')

**What this does:** predicts on the test set, prints precision/recall per class, draws a confusion matrix and lists the most common mix-ups (your error analysis). Expect confusion between visually similar classes such as Paper/Cardboard or Plastic/Miscellaneous Trash. Describe what you see in a short written analysis.

### 2.4 Fine-tune the Model

In [ ]:
# Fine-tuning: unfreeze the top layers of MobileNet so they adapt to waste images
base_model.trainable = True
for layer in base_model.layers[:-30]:   # keep all but the last 30 layers frozen
    layer.trainable = False

# Much smaller learning rate so we gently adjust, not destroy, the pretrained weights
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = cnn_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=cnn_callbacks
)

plot_history(history_ft, 'Fine-tuning:')
finetuned_acc = evaluate_cnn(cnn_model, test_dataset, 'Fine-tuned:')

print(f"Before fine-tuning: {stage1_acc:.3f}  |  After fine-tuning: {finetuned_acc:.3f}")

**What this does:** unfreezes the last 30 layers of the base model and trains them at a very low learning rate (1e-5), then compares accuracy before and after. If results do not improve, try a different number of unfrozen layers or a dropout of 0.4-0.5, and note what you tried.

## Part 3: Waste Description Classification

In this section, you will build a text classification model to categorize waste based on descriptions.

### 3.1 Preprocess Text Data

In [ ]:
# The pipeline from Section 1.3 already created everything we need:
# X_train / X_test (TF-IDF features), y_train / y_test (numeric labels)
print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("\nExample cleaned description:", train_df['clean_description'].iloc[0])
print("Its category:", train_df['category'].iloc[0])

This confirms the data is ready. The preprocessing was built in Section 1.3 and is reused here.

### 3.2 Implement Text Classification Model

In [ ]:
# Option A (traditional ML). We try two simple, fast models and compare them.
# Logistic Regression learns a weight per word for each category.
# Naive Bayes uses word-frequency probabilities.
models_to_try = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
}
print("Models to compare:", list(models_to_try.keys()))

**Why this choice:** the descriptions are short (about 5 words) and keyword-driven (e.g. "bottle", "cardboard"). TF-IDF with a linear model trains in seconds and is easy to explain. A transformer is usually not worth the extra cost for text this simple.

### 3.3 Train and Evaluate the Model

In [ ]:
results = {}
for name, model in models_to_try.items():
    model.fit(X_train, y_train)                      # training
    preds = model.predict(X_test)                    # prediction on unseen data
    results[name] = accuracy_score(y_test, preds)
    print(f"{name}: test accuracy = {results[name]:.4f}")

best_name = max(results, key=results.get)
text_model = models_to_try[best_name]
print("\nBest model:", best_name)

**What this does:** trains both models and keeps the more accurate one as `text_model`.

In [ ]:
# Detailed evaluation of the best model
y_pred = text_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{best_name} - Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.show()

# Error analysis: look at the actual misclassified descriptions
errors = test_df.copy()
errors['predicted'] = label_encoder.inverse_transform(y_pred)
errors = errors[errors['category'] != errors['predicted']]
print(f"Misclassified: {len(errors)} of {len(test_df)}")
display(errors[['description', 'category', 'predicted']].head(15))

**What this does:** prints accuracy per class, draws a confusion matrix and shows the descriptions the model got wrong. Look for patterns: many errors are probably descriptions with mixed signals (for example a material word that disagrees with the item), which is worth discussing in your analysis.

### 3.4 Create Classification Function

In [ ]:
def classify_waste_description(description):
    """
    Classifies a waste description into an appropriate category.

    Args:
        description (str): Text description of waste item

    Returns:
        str: Predicted waste category
    """
    cleaned = clean_text(description)              # same cleaning as training
    features = tfidf.transform([cleaned])          # same vectorizer as training
    label = text_model.predict(features)[0]        # numeric prediction
    return label_encoder.inverse_transform([label])[0]  # number -> category name


def classify_with_confidence(description):
    """Helper used by the integrated assistant: returns (category, confidence)."""
    features = tfidf.transform([clean_text(description)])
    probs = text_model.predict_proba(features)[0]
    idx = np.argmax(probs)
    return label_encoder.classes_[idx], float(probs[idx])

# Test it
for example in ["empty plastic water bottle", "old cotton t-shirt", "banana peel", "flattened shipping box"]:
    print(f"{example!r} -> {classify_waste_description(example)}")

**What this does:** wraps the whole text pipeline (clean, vectorise, predict, convert back to a name) into one function. The helper also returns a confidence score for Part 5.

## Part 4: Recycling Instruction Generation with RAG

In this section, you will implement a Retrieval-Augmented Generation (RAG) system to generate recycling instructions.

### 4.1 Preprocess Documents for Retrieval

In [ ]:
# The chunks and embeddings were built in Section 1.3 (chunk_df, chunk_embeddings).
# Here we turn them into a search function.
def retrieve(query, category=None, top_k=3):
    """
    Find the policy chunks most similar to the query.
    If a category is given, only search chunks for that category + General rules.
    """
    query_vec = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_embeddings @ query_vec        # cosine similarity (vectors are normalised)

    mask = np.ones(len(chunk_df), dtype=bool)
    if category is not None:
        mask = chunk_df['category'].isin([category, 'General']).values
    scores = np.where(mask, scores, -1)          # push excluded chunks to the bottom

    top_idx = np.argsort(scores)[::-1][:top_k]
    return chunk_df.iloc[top_idx].assign(score=scores[top_idx])

display(retrieve("how do I recycle a glass jar", category='Glass')[['text', 'score']])

**What this does:** compares the question's embedding to every chunk's embedding and returns the closest matches. Filtering by category means a "Glass" question never pulls in "Plastic" rules.

### 4.2 Implement RAG-based System

In [ ]:
from transformers import pipeline

# FLAN-T5 is a small instruction-following model that runs on a normal laptop or Colab.
generator = pipeline('text2text-generation', model='google/flan-t5-base')

def build_prompt(category, context_chunks):
    """Combine retrieved policy text and the request into one prompt."""
    context = '\n'.join(context_chunks)
    return (
        f"Use the city recycling policy below to write clear, step-by-step "
        f"recycling instructions for {category}. Mention what is accepted, "
        f"what is not accepted, and how to prepare the items.\n\n"
        f"Policy:\n{context}\n\nInstructions:"
    )

# Quick test
test_category = 'Plastic'
retrieved = retrieve(f"how to recycle {test_category}", category=test_category, top_k=3)
prompt = build_prompt(test_category, retrieved['text'].tolist())
output = generator(prompt, max_new_tokens=200, num_beams=4, no_repeat_ngram_size=3)
print(output[0]['generated_text'])

**What this does (the RAG idea):** instead of letting the language model guess, we first retrieve the real policy text and paste it into the prompt. The model then writes instructions grounded in Metro City's actual rules.

### 4.3 Adjust and Evaluate the System

In [ ]:
# Compare different generation settings on the same prompt
settings = {
    'Greedy (deterministic)':   dict(do_sample=False),
    'Beam search (4 beams)':    dict(do_sample=False, num_beams=4, no_repeat_ngram_size=3),
    'Sampling temp=0.7, top_p=0.9': dict(do_sample=True, temperature=0.7, top_p=0.9),
    'Sampling temp=1.2, top_p=0.95': dict(do_sample=True, temperature=1.2, top_p=0.95),
}
for name, params in settings.items():
    out = generator(prompt, max_new_tokens=150, **params)[0]['generated_text']
    print(f"=== {name} ===\n{out}\n")

**What this does:** shows how sampling settings change the output. Greedy and beam search are stable and factual, which suits policy instructions. A high temperature adds randomness and can invent details. Beam search is used in the final function.

In [ ]:
def overlap_score(generated, context_chunks):
    """Share of generated words that also appear in the retrieved policy (a groundedness proxy)."""
    gen_words = set(re.findall(r'[a-z]+', generated.lower()))
    ctx_words = set(re.findall(r'[a-z]+', ' '.join(context_chunks).lower()))
    return len(gen_words & ctx_words) / max(len(gen_words), 1)

eval_rows = []
for cat in sorted(text_df['category'].unique()):
    ctx = retrieve(f"how to recycle {cat}", category=cat, top_k=3)['text'].tolist()
    gen = generator(build_prompt(cat, ctx), max_new_tokens=200,
                    num_beams=4, no_repeat_ngram_size=3)[0]['generated_text']
    eval_rows.append({
        'category': cat,
        'overlap_with_policy': round(overlap_score(gen, ctx), 2),
        'mentions_category': cat.lower().split()[0] in gen.lower(),
        'generated': gen
    })
    print(f"--- {cat} ---\n{gen}\n")

eval_df = pd.DataFrame(eval_rows)
display(eval_df[['category', 'overlap_with_policy', 'mentions_category']])
print("Average overlap with policy text:", eval_df['overlap_with_policy'].mean().round(2))

**What this does:** generates instructions for all 9 categories, scores how closely each answer sticks to the retrieved policy, and checks whether it names the category. Also read the outputs yourself and note any that are vague, repetitive or wrong. That is the qualitative part of the evaluation.

### 4.4 Create Instruction Generation Function

In [ ]:
def generate_recycling_instructions(waste_category):
    """
    Generates detailed recycling instructions for a given waste category.

    Args:
        waste_category (str): Waste category

    Returns:
        str: Detailed recycling instructions
        list: Relevant policy documents
    """
    # 1) Retrieve the most relevant policy chunks for this category
    retrieved = retrieve(f"how to recycle {waste_category}", category=waste_category, top_k=3)

    # 2) Build the prompt and generate with the settings chosen in 4.3
    prompt = build_prompt(waste_category, retrieved['text'].tolist())
    instructions = generator(prompt, max_new_tokens=200,
                             num_beams=4, no_repeat_ngram_size=3)[0]['generated_text']

    # 3) Return the text and the names of the policies used
    policies_used = sorted(retrieved['policy_type'].unique().tolist())
    return instructions, policies_used

text_out, sources = generate_recycling_instructions('Glass')
print(text_out)
print("Sources:", sources)

## Part 5: Integrated Waste Management Assistant

In this section, you will integrate all three models into a unified waste management assistant.

### 5.1 Design Integration Architecture

**Architecture**

```
Input (image OR text)
        |
        v
+-------------------------+      +--------------------------------+
| image -> CNN            |      | text -> TF-IDF + classifier    |
| (MobileNetV2)           |      | (Logistic Regression / NB)     |
+-----------+-------------+      +---------------+----------------+
            \                                    /
             v                                  v
              predicted category + confidence
                          |
                          v
         RAG: retrieve policy chunks for that category
                          |
                          v
         FLAN-T5 generates recycling instructions
                          |
                          v
   Output dict: category, confidence, instructions, policies
```

Both classifiers produce the same output, a category name plus a confidence score. That shared interface lets one RAG component serve both input types.

### 5.2 Implement Integrated Assistant

In [ ]:
def predict_image_category(image_input):
    """Accepts a file path OR a numpy array; returns (category, confidence)."""
    if isinstance(image_input, (str, pathlib.Path)):
        img = tf.keras.utils.load_img(image_input, target_size=(IMG_HEIGHT, IMG_WIDTH))
        img_array = tf.keras.utils.img_to_array(img)
    else:
        img_array = np.array(image_input, dtype='float32')
    img_array = np.expand_dims(img_array, axis=0)   # model expects a batch: (1, 224, 224, 3)

    probs = cnn_model.predict(img_array, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx])


def waste_management_assistant(input_data, input_type="image"):
    """
    Integrated waste management assistant that processes either images or text descriptions
    and returns waste classification and recycling instructions.

    Args:
        input_data: Either an image file path/array or a text description
        input_type (str): Type of input - "image" or "text"

    Returns:
        dict: Dictionary containing waste category, confidence, and recycling instructions
    """
    # Step 1: classify using the right model
    if input_type == "image":
        category, confidence = predict_image_category(input_data)
    elif input_type == "text":
        category, confidence = classify_with_confidence(input_data)
    else:
        raise ValueError("input_type must be 'image' or 'text'")

    # Step 2: generate instructions with RAG
    instructions, policies = generate_recycling_instructions(category)

    # Step 3: return everything in one dictionary
    return {
        'input_type': input_type,
        'category': category,
        'confidence': round(confidence, 3),
        'instructions': instructions,
        'policies_used': policies,
    }

# Demo
print(waste_management_assistant("a crushed aluminum soda can", input_type="text"))

### 5.3 Evaluate the Integrated System

In [ ]:
# --- Text test: 200 random descriptions from the held-out test set ---
text_sample = test_df.sample(200, random_state=42)
text_correct = 0
for _, row in text_sample.iterrows():
    result = waste_management_assistant(row['description'], input_type="text")
    text_correct += (result['category'] == row['category'])
print(f"Text assistant accuracy: {text_correct / len(text_sample):.3f}")

# --- Image test: 2 batches (64 images) from the test dataset ---
img_correct, img_total = 0, 0
for images, labels in test_dataset.take(2):
    for img, lab in zip(images.numpy(), labels.numpy()):
        true_cat = class_names[np.argmax(lab)]
        category, _ = predict_image_category(img)   # array input
        img_correct += (category == true_cat)
        img_total += 1
print(f"Image assistant accuracy: {img_correct / img_total:.3f}")

# --- End-to-end check: does the output look complete and on-topic? ---
sample_result = waste_management_assistant("old cotton t-shirt", input_type="text")
print("\nExample full output:")
for k, v in sample_result.items():
    print(f"{k}: {v}")

**What this does:** tests the full pipeline on unseen data (200 text descriptions and 64 images) and prints one complete output so you can check the instructions read sensibly.

**Final analysis (write your own after running):** compare image vs text accuracy, say where the system struggles, and name what you would improve (e.g. a larger language model, more image augmentation, or confidence thresholds that ask the user for another photo).